In [ ]:
import funciones as f
from PIL import Image
import numpy as np
import pandas as pd
from scipy.ndimage import median_filter, binary_closing, binary_opening
from skimage.morphology import disk, remove_small_holes
from scipy import ndimage
import matplotlib.pyplot as plt

In [ ]:
#--------------------------------------------------------------
# Paso 0: cargar imagen
#--------------------------------------------------------------
img_raw = f.cargar_imagen('img/pensamientos.jpg')
# img_raw = funciones.cargar_imagen(r"img/flores de lupino.png")
h, w, c = img_raw.shape
#--------------------------------------------------------------


#--------------------------------------------------------------
# Paso 1: reducir cantidad de colores
#--------------------------------------------------------------

img_cuantizada, paleta_img, labels, colores_paleta = f.cuantizar_imagen(img_raw, n_colores=16)

#guardar resultado paso 1
# Image.fromarray(img_cuantizada).save('output/paso2/filtrada_0.png')
#--------------------------------------------------------------


#--------------------------------------------------------------
# Paso 2: eliminar ruido de 1px
#--------------------------------------------------------------
img_cuantizada_filtrada, labels_filtrados = f.eliminar_1px(img_cuantizada, labels, colores_paleta)
#guardar resultado paso 2
# Image.fromarray(img_cuantizada_filtrada).save('output/paso2/filtrada_1.png')
#--------------------------------------------------------------


#--------------------------------------------------------------
# Paso 3: segmentar y reducir regiones pequeñas
#--------------------------------------------------------------
porcentaje = 0.1
area_minima_est = f.calcular_area_minima(h, w, porcentaje, area_minima=100)

mapa_regiones, df_regiones = f.segmentar_regiones(
    img_cuantizada_filtrada,
    labels_filtrados,
    colores_paleta
)

#filtrar regiones pequeñas
mapa_regiones_limpio, df_regiones_limpio, stats_filtrado = f.filtrar_regiones_pequenas(
    mapa_regiones,
    df_regiones,
    area_minima_est
)

In [ ]:
import numpy as np
import pandas as pd
from scipy import ndimage


def marcar_regiones_delgadas(
    mapa_regiones,
    df_regiones,
    grosor_max=10,
    color_rojo_id=16
):
    """
    Detecta regiones con grosor <= grosor_max
    y cambia su color_id en el DataFrame.

    Parámetros
    ----------
    mapa_regiones : np.array (H,W)
        Cada pixel contiene el id de región.

    df_regiones : pd.DataFrame
        Debe contener:
        - region_id
        - color_id

    grosor_max : int
        Grosor máximo permitido.

    color_rojo_id : int
        ID del color rojo en la paleta.

    Retorna
    -------
    df_actualizado : pd.DataFrame
    """

    df_actualizado = df_regiones.copy()

    regiones_unicas = np.unique(mapa_regiones)

    for region_id in regiones_unicas:

        mask = mapa_regiones == region_id

        # distancia al borde
        dist = ndimage.distance_transform_edt(mask)

        # grosor aproximado
        grosor = dist * 2

        # grosor máximo interno de la región
        grosor_region = grosor.max()

        # si la región completa es delgada
        if grosor_region <= grosor_max:

            df_actualizado.loc[
                df_actualizado["region_id"] == region_id,
                "color_id"
            ] = color_rojo_id

    return df_actualizado

In [ ]:
df_nuevo = marcar_regiones_delgadas(
    mapa_regiones_limpio,
    df_regiones_limpio,
    grosor_max=8,
    color_rojo_id=17
)

In [ ]:
colores_paleta = np.append(colores_paleta, [[255, 0, 0]], axis=0)

In [ ]:
mapa_colores = dict(zip(df_nuevo['region_id'], df_nuevo['color_id']))

img_regiones_filtrado = np.vectorize(mapa_colores.get)(mapa_regiones_limpio.reshape(-1))

img_regiones_filtrado = img_regiones_filtrado.reshape(h, w)

img_regiones_filtradas = colores_paleta[img_regiones_filtrado - 1]

# Image.fromarray(img_regiones_filtradas).save('output/paso3_2/regiones_filtrada_final.png')



In [ ]:
Image.fromarray(img_regiones_filtradas.astype(np.uint8)).save('output/paso3_2/regiones_filtrada_final.png')